# Parent-Document Retrieval: Retrieve Small, Return Useful Context

Small chunks can be precise search units but omit context needed to interpret a sentence. Large chunks preserve context but can be harder to search and consume more of the answer model's context window. **Parent-document retrieval** indexes small child chunks while retaining a link to a larger parent section. The system searches a child, then returns the parent or a bounded context window around the match.

This notebook builds the child-to-parent mapping explicitly, retrieves focused children with a transparent lexical baseline, expands them to parent sections or nearby chunks, and compares context size with evidence coverage. Two SVG charts visualize the trade-offs. The code uses only the Python standard library.

## Learning goals

- Explain the child-chunk/parent-context pattern and why the two units can differ.
- Preserve stable IDs, parent links, and order for retrieved children.
- Expand a match to a full parent or bounded local window while deduplicating results.
- Compare focused child text with expanded context using evidence coverage and text size.
- Identify mapping, freshness, ordering, duplicate, and access-control failure modes.
- Interpret charts showing context growth and evidence coverage.

## 1. The granularity trade-off

A retrieval unit has to balance two goals:

- **Search precision:** small focused chunks often make it easier to match a specific information need.
- **Interpretive context:** surrounding sentences can explain pronouns, conditions, exceptions, and relationships.

If we return only a sentence such as “It must be renewed within thirty days,” the passage may not identify what “it” refers to. Returning an entire long handbook can bury the relevant fact in unrelated material and use excessive context. Parent-document retrieval separates the **search unit** from the **return unit**:

1. Split a source into focused child chunks.
2. Store a stable parent ID and child position/offset for every child.
3. Search the children.
4. Expand a selected child to a parent section or nearby window.
5. Deduplicate, preserve provenance, and respect a context budget.

A parent is not necessarily an entire source document. A heading section, paragraph group, page, or other coherent span may be a better parent.

## 2. Create parent sections and child chunks

The four parents below are coherent sections. Their child chunks are small searchable passages with stable IDs and explicit positions. Metadata such as source, title, version, and access scope can accompany each record in a real index.

In [ ]:
parents = {
    'RETURNS': {'source': 'shopping_policy', 'title': 'Returning a product',
                'text': 'Unopened products may be returned within thirty days of delivery. Start a return request with the order number. After the returned item is inspected, eligible refunds go back to the original payment method. Items marked final sale are not eligible for return.'},
    'SHIPPING': {'source': 'shipping_guide', 'title': 'Shipping and delivery',
                 'text': 'Standard shipping usually takes five to seven business days. Express shipping is available at checkout. Delivery estimates begin after the order ships and may be longer during regional disruptions.'},
    'ACCOUNT': {'source': 'account_help', 'title': 'Resetting account access',
                'text': 'To reset your password, choose Forgot password on the sign-in page. A reset link is sent to the email address registered to the account. If you cannot access that email, contact support to verify account ownership.'},
    'LEASE': {'source': 'tenant_handbook', 'title': 'Lease security deposit',
              'text': 'The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement. Contact the property manager if the agreement and payment notice disagree.'},
}

children = [
    {'child_id': 'RET-1', 'parent_id': 'RETURNS', 'position': 0, 'text': 'Unopened products may be returned within thirty days of delivery.'},
    {'child_id': 'RET-2', 'parent_id': 'RETURNS', 'position': 1, 'text': 'Start a return request with the order number.'},
    {'child_id': 'RET-3', 'parent_id': 'RETURNS', 'position': 2, 'text': 'After inspection, eligible refunds go back to the original payment method.'},
    {'child_id': 'RET-4', 'parent_id': 'RETURNS', 'position': 3, 'text': 'Items marked final sale are not eligible for return.'},
    {'child_id': 'SHIP-1', 'parent_id': 'SHIPPING', 'position': 0, 'text': 'Standard shipping usually takes five to seven business days.'},
    {'child_id': 'SHIP-2', 'parent_id': 'SHIPPING', 'position': 1, 'text': 'Express shipping is available at checkout.'},
    {'child_id': 'SHIP-3', 'parent_id': 'SHIPPING', 'position': 2, 'text': 'Delivery estimates begin after the order ships and may be longer during regional disruptions.'},
    {'child_id': 'ACC-1', 'parent_id': 'ACCOUNT', 'position': 0, 'text': 'To reset your password, choose Forgot password on the sign-in page.'},
    {'child_id': 'ACC-2', 'parent_id': 'ACCOUNT', 'position': 1, 'text': 'A reset link is sent to the email address registered to the account.'},
    {'child_id': 'ACC-3', 'parent_id': 'ACCOUNT', 'position': 2, 'text': 'If you cannot access that email, contact support to verify account ownership.'},
    {'child_id': 'LEA-1', 'parent_id': 'LEASE', 'position': 0, 'text': 'The security deposit is due when the lease is signed.'},
    {'child_id': 'LEA-2', 'parent_id': 'LEASE', 'position': 1, 'text': 'The deposit amount is listed in the signed lease agreement.'},
    {'child_id': 'LEA-3', 'parent_id': 'LEASE', 'position': 2, 'text': 'Contact the property manager if the agreement and payment notice disagree.'},
]

children_by_parent = {}
for child in children:
    if child['parent_id'] not in parents:
        raise KeyError(f"Missing parent: {child['parent_id']}")
    children_by_parent.setdefault(child['parent_id'], []).append(child)
for group in children_by_parent.values():
    group.sort(key=lambda item: item['position'])

assert len({child['child_id'] for child in children}) == len(children)
print(f'{len(parents)} parents; {len(children)} child chunks')
print('Returns child order:', [c['child_id'] for c in children_by_parent['RETURNS']])

The mapping is explicit rather than inferred from IDs. Production systems should validate that IDs are unique, every child references an existing parent, positions/offsets are valid, and parent and child records refer to the same content version. If a parent changes, update its children and embeddings consistently.

## 3. Retrieve focused child chunks

To keep the retrieval mechanics easy to inspect, this notebook uses a simple score: the number of distinct query tokens shared with a child. It is not BM25, dense similarity, or a production recommendation. Parent expansion can follow any retriever—sparse, dense, hybrid, or database search—because the important requirement is that each hit retains its `child_id` and `parent_id`.

In [ ]:
import re

STOP_WORDS = {'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'how', 'i', 'in', 'is', 'it', 'of', 'on', 'or', 'the', 'to', 'what', 'when', 'where', 'within', 'with'}

def tokenize(text):
    return set(re.findall(r'[a-z0-9]+', text.lower())) - STOP_WORDS

def retrieve_children(query, k=5):
    if k < 0:
        raise ValueError('k must be non-negative')
    results = []
    for child in children:
        child_score = len(tokenize(query) & tokenize(child['text']))
        if child_score > 0:
            results.append({**child, 'score': child_score})
    return sorted(results, key=lambda item: (-item['score'], item['child_id']))[:k]

query = 'What is the return window and are final sale items eligible?'
child_results = retrieve_children(query, k=5)
for rank, child in enumerate(child_results, start=1):
    print(f"{rank}. {child['child_id']} score={child['score']} parent={child['parent_id']} | {child['text']}")

`RET-1` contains the return window; `RET-4` contains the final-sale exclusion. Searching the whole parent might find both details but make it harder to identify which text matched. Searching children yields focused hits. The key is to expand only after retrieval and preserve the matched child IDs so the context remains traceable.

## 4. Expand child hits to parent sections

The simplest policy maps every child hit to its entire parent. We deduplicate by parent ID so a section matched by multiple children is returned once, while recording all matched children and the best child score.

In [ ]:
def expand_to_parents(child_results):
    expanded = {}
    for child in child_results:
        parent_id = child['parent_id']
        if parent_id not in parents:
            raise KeyError(f"Child {child['child_id']} references missing parent {parent_id}")
        if parent_id not in expanded:
            expanded[parent_id] = {
                'parent_id': parent_id,
                **parents[parent_id],
                'matched_children': [],
                'best_child_score': child['score'],
            }
        expanded[parent_id]['matched_children'].append(child['child_id'])
        expanded[parent_id]['best_child_score'] = max(expanded[parent_id]['best_child_score'], child['score'])
    return sorted(expanded.values(), key=lambda item: (-item['best_child_score'], item['parent_id']))

parent_results = expand_to_parents(child_results)
for item in parent_results:
    print(f"{item['parent_id']} {item['title']} matched={item['matched_children']}")
    print(' ', item['text'])

Parent deduplication saves context budget when several children from one section match. Retain the match provenance: a downstream generator, reviewer, or citation renderer may need to know exactly which children supported the result. If parent sections are very large, returning the complete parent can still be wasteful.

## 5. Alternative: expand to a local child window

A local window returns the matched child plus a fixed number of neighbors from the same parent. This can include nearby definitions and qualifiers without returning the entire section. It depends on a reliable ordering field; sorting opaque IDs is not a safe substitute for source positions or offsets.

In [ ]:
def expand_window(child_result, radius=1):
    if radius < 0:
        raise ValueError('radius must be non-negative')
    group = children_by_parent[child_result['parent_id']]
    index_by_id = {child['child_id']: i for i, child in enumerate(group)}
    child_id = child_result['child_id']
    if child_id not in index_by_id:
        raise KeyError(f'Unknown child ID: {child_id}')
    center = index_by_id[child_id]
    start = max(0, center - radius)
    end = min(len(group), center + radius + 1)
    return group[start:end]

matched = next(item for item in retrieve_children('unopened products return thirty days delivery', k=10)
               if item['child_id'] == 'RET-1')
for radius in (0, 1, 2, 3):
    window = expand_window(matched, radius)
    print(f'radius={radius}: {[child["child_id"] for child in window]}')
    print(' ', ' '.join(child['text'] for child in window))

The radius is bounded by the parent's beginning and end. At a boundary, a radius of one may return fewer than three children. A larger window adds context but may also add unrelated text; use the question's required evidence and the target model's context budget to select a policy.

## 6. Merge overlapping windows and keep provenance

If two nearby children match, expanding each independently can duplicate text. The helper below unions selected positions for each parent, keeps source order, and retains the IDs of children that triggered expansion.

In [ ]:
def merge_windows(child_results, radius=1):
    if radius < 0:
        raise ValueError('radius must be non-negative')
    positions_by_parent = {}
    matches_by_parent = {}
    for result in child_results:
        parent_id = result['parent_id']
        if parent_id not in children_by_parent:
            raise KeyError(f'No child sequence for parent {parent_id}')
        group = children_by_parent[parent_id]
        index_by_id = {child['child_id']: i for i, child in enumerate(group)}
        child_id = result['child_id']
        if child_id not in index_by_id:
            raise KeyError(f'Unknown child ID: {child_id}')
        center = index_by_id[child_id]
        start = max(0, center - radius)
        end = min(len(group), center + radius + 1)
        positions_by_parent.setdefault(parent_id, set()).update(range(start, end))
        matches_by_parent.setdefault(parent_id, set()).add(child_id)

    output = []
    for parent_id, positions in positions_by_parent.items():
        group = children_by_parent[parent_id]
        selected = [group[i] for i in sorted(positions)]
        output.append({
            'parent_id': parent_id,
            'title': parents[parent_id]['title'],
            'matched_children': sorted(matches_by_parent[parent_id]),
            'context_children': selected,
            'text': ' '.join(child['text'] for child in selected),
        })
    return sorted(output, key=lambda item: item['parent_id'])

merged_contexts = merge_windows(child_results, radius=1)
for context in merged_contexts:
    print(context['parent_id'], 'matches:', context['matched_children'])
    print(' returned children:', [c['child_id'] for c in context['context_children']])
    print(' text:', context['text'])

Merging by position prevents repeated neighboring text while preserving source order. In a production implementation, character/token offsets can support more precise windows. Include a maximum token budget: many matches in a long parent should not automatically cause the whole source to be returned.

## 7. Compare context size and evidence coverage

For the return-policy example, the information need asks for both the return window and whether final-sale items are eligible. `RET-1` has the time window and `RET-4` has the exclusion. We compare three policies around `RET-1`: child only, a one-child neighborhood, and the full parent. Word count is only a rough size proxy; production token budgets should use the generation model's tokenizer.

In [ ]:
def word_count(text):
    return len(re.findall(r'\b\w+\b', text))

matched_ret = next(item for item in retrieve_children('return window days final sale eligible', k=10)
                   if item['child_id'] == 'RET-1')
child_text = matched_ret['text']
window_text = ' '.join(child['text'] for child in expand_window(matched_ret, radius=1))
parent_text = parents['RETURNS']['text']
contexts = {'child only': child_text, 'local window': window_text, 'full parent': parent_text}

requirements = {
    'return timing': {'thirty', 'days', 'delivery'},
    'final-sale exclusion': {'final', 'sale', 'eligible', 'return'},
}

def evidence_coverage(text):
    terms = tokenize(text)
    hits = {name: bool(required & terms) for name, required in requirements.items()}
    return sum(hits.values()) / len(hits), hits

sizes = {}
coverages = {}
for name, text in contexts.items():
    coverage, checks = evidence_coverage(text)
    sizes[name] = word_count(text)
    coverages[name] = coverage
    print(name, 'words=', sizes[name], 'coverage=', round(coverage, 2), 'checks=', checks)

The evidence check is a deliberately simple keyword proxy, not a semantic metric. It illustrates that a larger return unit can include a qualifier omitted from a focused match. In a real evaluation, label required facts and judge whether returned passages actually support them; a keyword hit alone can be misleading.

## 8. Charts: context size and evidence coverage

The charts below are generated as SVG from the measured values. SVG renders inline in a Jupyter notebook when IPython is available and scales cleanly. No plotting dependency is required; outside Jupyter the cells still validate the SVG and report that it was generated.

In [ ]:
from html import escape
from xml.etree import ElementTree

try:
    from IPython.display import SVG, display
except ImportError:
    SVG = None
    display = None

def show_svg(svg_text):
    ElementTree.fromstring(svg_text)
    if SVG is not None and display is not None:
        display(SVG(svg_text))
    else:
        print('SVG generated and validated; run in Jupyter/IPython to display inline.')

def bar_chart_svg(title, values, suffix=' words'):
    width, left, top, row_height = 780, 190, 62, 44
    max_width = 470
    max_value = max(max(values.values()), 1)
    height = top + row_height * len(values) + 18
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Arial" font-size="19" font-weight="bold" fill="#111827">{escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values.items()):
        y = top + index * row_height
        bar_width = max_width * value / max_value
        parts.append(f'<text x="{left-12}" y="{y+18}" text-anchor="end" font-family="Arial" font-size="13" fill="#374151">{escape(label)}</text>')
        parts.append(f'<rect x="{left}" y="{y}" width="{bar_width:.1f}" height="23" rx="4" fill="#2563eb"/>')
        parts.append(f'<text x="{left+bar_width+8:.1f}" y="{y+17}" font-family="Arial" font-size="12" fill="#111827">{value}{escape(suffix)}</text>')
    parts.append('</svg>')
    return ''.join(parts)

show_svg(bar_chart_svg('Approximate context size by return policy', sizes))

The full parent is larger because it includes the order-number requirement, refund behavior, and final-sale exclusion. That additional text can be useful or wasteful depending on the question. Context size alone is not a quality measure.

In [ ]:
def scatter_chart_svg(title, points):
    width, height = 780, 410
    left, right, top, bottom = 75, 30, 58, 72
    plot_width, plot_height = width - left - right, height - top - bottom
    max_x = max(max(x for x, _ in points.values()), 1)
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Arial" font-size="19" font-weight="bold" fill="#111827">{escape(title)}</text>',
    ]
    for tick in (0, 0.25, 0.5, 0.75, 1.0):
        y = top + plot_height * (1 - tick)
        parts.append(f'<line x1="{left}" y1="{y:.1f}" x2="{width-right}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        parts.append(f'<text x="{left-9}" y="{y+4:.1f}" text-anchor="end" font-family="Arial" font-size="10" fill="#6b7280">{tick:.0%}</text>')
    parts.append(f'<text x="{width/2}" y="{height-20}" text-anchor="middle" font-family="Arial" font-size="12" fill="#374151">Approximate context size (words)</text>')
    for label, (x_value, y_value) in points.items():
        x = left + plot_width * x_value / max_x
        y = top + plot_height * (1 - y_value)
        parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="7" fill="#7c3aed"/>')
        parts.append(f'<text x="{x+10:.1f}" y="{y-8:.1f}" font-family="Arial" font-size="11" fill="#374151">{escape(label)}</text>')
    parts.append('</svg>')
    return ''.join(parts)

coverage_points = {name: (sizes[name], coverages[name]) for name in contexts}
show_svg(scatter_chart_svg('Evidence coverage versus approximate context size', coverage_points))

The upper-left is desirable only when the returned context truly supports all required facts. Here, the chart uses keyword-based labels; it is a teaching diagnostic, not a universal optimization objective. A larger parent can include irrelevant or conflicting content, and a narrow window can omit a distant exception.

## 9. Choosing an expansion strategy

| Return policy | Strength | Risk |
| --- | --- | --- |
| Child only | Small, focused, low context cost | May omit qualifications and references |
| Full parent | Broad section context and distant qualifiers | May be large, noisy, or contradictory |
| Local window | Bounded surrounding text | May miss a distant exception |
| Selected sibling chunks | Targeted details with low duplication | Requires reliable reranking/selection logic |

Useful decision factors include query type, parent size, downstream token budget, whether exceptions may be distant, and the cost of missing a qualifier. One sensible default is to retrieve focused children, expand to a small local window, and increase context only when the query requires broader relationships or evaluation shows that local windows omit necessary evidence.

## 10. Operational and safety considerations

- **Stable mapping:** missing or stale parent links should raise a visible data-quality error, not silently fall back to unrelated content.
- **Version consistency:** child embeddings and parent text must refer to compatible source versions. Update them atomically or store version identifiers.
- **Ordering:** use offsets or explicit positions; do not infer adjacency from opaque IDs.
- **Deduplication and budget:** repeated child hits should not repeat entire parent text. Track token cost using the target model tokenizer.
- **Access controls:** verify authorization for the parent during expansion as well as for the child during search. Child access does not automatically imply parent access.
- **Context integrity:** expanded text can contain stale, conflicting, or unrelated statements. Preserve source metadata and apply source/version precedence rules.
- **Evaluation:** judge evidence coverage and answer grounding, not only whether the retrieved child matched the query.

Parent expansion is a retrieval stage, not a guarantee of correctness. It cannot recover information missing from the source collection.

## 11. Exercises

1. Query for refund timing. Which child matches, and does the parent add needed process context?
2. Query for final-sale eligibility. Does expanding `RET-4` include the return window as related context?
3. Retrieve two adjacent children in one parent and merge radius-1 windows. Confirm each child appears once and source order is preserved.
4. Create a parent with ten children. Compare radii 0, 1, and 2 by evidence coverage and token count.
5. Simulate a child referencing a missing parent. Verify expansion fails explicitly.
6. Add parent and child version IDs. Design a check that prevents expansion to a mismatched version.
7. Add an access-scope field and show why authorization must be checked at both retrieval and expansion.
8. Build relevance labels for several questions and compare child-only, local-window, and full-parent policies using a real tokenizer and reviewed evidence judgments.

### Optional extension: structural checks

These assertions verify mapping, unique parent expansion, child provenance, stable ordering, and explicit rejection of an invalid context radius.

In [ ]:
assert all(child['parent_id'] in parents for child in children)
assert len({item['parent_id'] for item in parent_results}) == len(parent_results)
assert all(item['matched_children'] for item in parent_results)
assert retrieve_children(query, k=5) == retrieve_children(query, k=5)

window_ids = [item['child_id'] for item in expand_window(matched, radius=1)]
ordered_ids = [item['child_id'] for item in children_by_parent[matched['parent_id']]
               if item['child_id'] in window_ids]
assert window_ids == ordered_ids

try:
    expand_window(matched, radius=-1)
except ValueError:
    print('Negative radius rejected as expected.')
else:
    raise AssertionError('Negative radius should raise ValueError')

print('Parent mapping, deduplication, ordering, and input checks passed.')

## Summary

Parent-document retrieval searches focused child chunks but can return a larger parent section or a bounded local window to restore useful context. Reliable expansion depends on stable child/parent IDs, ordering or offsets, version consistency, deduplication, provenance, and access checks. Larger context can restore conditions and exceptions, but costs more and may introduce noise or conflict.

Compare child-only, local-window, and full-parent policies using evidence-coverage judgments and the actual generation model's token budget. Do not assume that more text is better; inspect whether the returned context supports the user's request. The next notebook moves to **advanced retrieval**, including routing, reranking, and iterative or corrective approaches.